# The Rest of the Iceberg — the native-kernel cut

Same exploration as the
[companion notebook](the-rest-of-the-iceberg.ipynb), on the
**Quarb Jupyter kernel**: every cell *is* a query — no `%%` fences,
no quoting — with tab-completion over the mounted tree. Directives
(`%mount`, `%use`, `%docs`, `%translate`, `%python`) manage state.

Setup:

```
pip install "quarb[jupyter]"
python -m quarb.kernel install
```

(on a distribution that manages its system Python, inside a
virtual environment, with Jupyter launched from it), then open this notebook with the **Quarb** kernel, with
`titanic.csv` and `titanic-birth-records.json` alongside.

In [1]:
%mount titanic.csv titanic-birth-records.json

mounted: titanic+titanic-birth-records (children: /titanic, /titanic-birth-records)


In [2]:
/titanic-birth-records/archives/*:name

gro
irish_parishes
kyrkobocker
actes_de_naissance
us_state_records
ottoman_nufus
matricula_baptizatorum
metricheskie_knigi

In [3]:
/titanic-birth-records/archives/* | .a(:name) | //passenger_id @| group($.a) | count | .n | %.

{'a': 'gro', 'n': 422}
{'a': 'irish_parishes', 'n': 124}
{'a': 'kyrkobocker', 'n': 166}
{'a': 'actes_de_naissance', 'n': 27}
{'a': 'us_state_records', 'n': 286}
{'a': 'ottoman_nufus', 'n': 80}
{'a': 'matricula_baptizatorum', 'n': 34}
{'a': 'metricheskie_knigi', 'n': 7}

## Naming the vocabulary

A `def` names a fragment of query text; invocations expand at parse
time. A cell holding only definitions (and `#` comments) extends
the **session's fragment table** — its names resolve in every later
cell, so the notebook states its vocabulary once, then speaks it.
`&record` is the birth record for the row at hand (`$$0` names the
driving row from inside the join condition); `&gregorian` converts
the Russian Empire's Julian dates:

In [4]:
# the exploration's vocabulary
def &record: /titanic-birth-records//*[::passenger_id = $$0::PassengerId] ;
def &records: /titanic-birth-records//*[::passenger_id] ;
# Julian to Gregorian, 1900-1917
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);

defined: &record, &records, &gregorian


In [5]:
/titanic/row <=>? &record | [!$$1::passenger_id] | ::Name @| [..6]

Vestrom, Miss. Hulda Amanda Adolfina
Hewlett, Mrs. (Mary D Kingcome) 
Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)
Todoroff, Mr. Lalio
Uruchurtu, Don. Manuel E
Vander Planke, Miss. Augusta Maria

In [6]:
&records | [::passenger_id > 891] @| count

373

## Recovering the missing ages

The eight-branch ladder normalizes eight date conventions into one
birth year — `&gregorian` comes from the vocabulary cell above:

In [7]:
/titanic/row[!::Age] <=> &record
 | .born(( $$1::year_rumi      ? $$1::year_rumi + 584
         : $$1::quarter        ? ($$1::quarter | s/ Q.*$//) * 1
         : $$1::date           ? ($$1::date | s/^.*\s//) * 1
         : $$1::född           ? ($$1::född | s/-.*$//) * 1
         : $$1::né_le          ? ($$1::né_le | s/-.*$//) * 1
         : $$1::date_of_birth  ? ($$1::date_of_birth | s/-.*$//) * 1
         : $$1::natus          ? ($$1::natus | s/-.*$//) * 1
         : $$1::nata           ? ($$1::nata | s/-.*$//) * 1
         : (($$1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | %(::Name; born = $.born; age_in_1912 = 1912 - $.born) @| [..10]

{'Name': 'Moran, Mr. James', 'born': 1889, 'age_in_1912': 23}
{'Name': 'Williams, Mr. Charles Eugene', 'born': 1875, 'age_in_1912': 37}
{'Name': 'Masselmani, Mrs. Fatima', 'born': 1877, 'age_in_1912': 35}
{'Name': 'Emir, Mr. Farred Chehab', 'born': 1881, 'age_in_1912': 31}
{'Name': 'O\'Dwyer, Miss. Ellen "Nellie"', 'born': 1894, 'age_in_1912': 18}
{'Name': 'Spencer, Mrs. William Augustus (Marie Eugenie)', 'born': 1888, 'age_in_1912': 24}
{'Name': 'Glynn, Miss. Mary Agatha', 'born': 1895, 'age_in_1912': 17}
{'Name': 'Mamee, Mr. Hanna', 'born': 1885, 'age_in_1912': 27}
{'Name': 'Rogers, Mr. William John', 'born': 1871, 'age_in_1912': 41}
{'Name': 'Lennon, Mr. Denis', 'born': 1872, 'age_in_1912': 40}

## The sisters nobody's ticket reveals

In [8]:
/titanic-birth-records//födda/*
 <=> /titanic/row[::PassengerId = $$0::passenger_id]
 <=> /titanic-birth-records//födda/*[::fader = $$0::fader
       && ::moder = $$0::moder && ::passenger_id > $$0::passenger_id]
 | [$$1::Name *= "Mrs."]
 | %(aboard_as = $$1::Name; sibling = $$2::namn)

{'aboard_as': 'Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson)', 'sibling': 'Gustaf Joel Johansson'}
{'aboard_as': 'Ahlin, Mrs. Johan (Johanna Persdotter Larsson)', 'sibling': 'Bengt Edvin Larsson'}
{'aboard_as': 'Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)', 'sibling': 'Anders Vilhelm Gustafsson'}

## Coming from jq?

`%translate` renders a jq (or XPath, or SQL) query as Quarb text —
useful while the muscle memory transfers:

In [9]:
%translate jq .archives.gro.entries[] | select(.quarter == "1890 Q1") | .surname

/archives/gro/entries/*[/quarter:: = "1890 Q1"]/surname::


## A macro that asks the data

Fragments are textual; *macros* run at expansion time, and the
data-aware form `&name!` may read the mounted dataset while
expanding. The classic showpiece — a pivot table whose columns are
the distinct values of a column, something no fixed query can
spell — mean fare by class, with a margin:

In [10]:
macro &pivot!($col, $of):
      /data/titanic/row | $col @| unique @| sort
        | "| .\"${$_}\"(^/titanic/row[${$col} = ${$_}]${$of} @| mean)";
    ^ | &pivot!(::Pclass, ::Fare)
      | .All(^/titanic/row::Fare @| mean) | %.

{'1': 84.15468749999992, '2': 20.66218315217391, '3': 13.675550101832997, 'All': 32.2042079685746}

## The escape hatch

`%python` cells are ordinary Python — `_` holds the last query
result, `session` is the live session (the fragment table
included, so `session.run` still speaks `&gregorian`), and `.df`
turns a record stream into a DataFrame. The enriched table, then
the finding:

In [11]:
/titanic/row <=>? &record
 | .born(( $$1::year_rumi      ? $$1::year_rumi + 584
         : $$1::quarter        ? ($$1::quarter | s/ Q.*$//) * 1
         : $$1::date           ? ($$1::date | s/^.*\s//) * 1
         : $$1::född           ? ($$1::född | s/-.*$//) * 1
         : $$1::né_le          ? ($$1::né_le | s/-.*$//) * 1
         : $$1::date_of_birth  ? ($$1::date_of_birth | s/-.*$//) * 1
         : $$1::natus          ? ($$1::natus | s/-.*$//) * 1
         : $$1::nata           ? ($$1::nata | s/-.*$//) * 1
         : (($$1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | %(id = ::PassengerId * 1; ::Sex;
     pclass = ::Pclass * 1; survived = ::Survived * 1;
     age = (( ::Age ? ::Age * 1 : 1912 - $.born ));
     age_source = (( ::Age ? "manifest" : $.born ? "records" : "unknown" )))
 @| [..5]

{'id': 1, 'Sex': 'male', 'pclass': 3, 'survived': 0, 'age': 22, 'age_source': 'manifest'}
{'id': 2, 'Sex': 'female', 'pclass': 1, 'survived': 1, 'age': 38, 'age_source': 'manifest'}
{'id': 3, 'Sex': 'female', 'pclass': 3, 'survived': 1, 'age': 26, 'age_source': 'manifest'}
{'id': 4, 'Sex': 'female', 'pclass': 1, 'survived': 1, 'age': 35, 'age_source': 'manifest'}
{'id': 5, 'Sex': 'male', 'pclass': 3, 'survived': 0, 'age': 35, 'age_source': 'manifest'}

In [12]:
%python
full = session.run("""
/titanic/row <=>? /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | .born(( $$1::year_rumi      ? $$1::year_rumi + 584
         : $$1::quarter        ? ($$1::quarter | s/ Q.*$//) * 1
         : $$1::date           ? ($$1::date | s/^.*\s//) * 1
         : $$1::född           ? ($$1::född | s/-.*$//) * 1
         : $$1::né_le          ? ($$1::né_le | s/-.*$//) * 1
         : $$1::date_of_birth  ? ($$1::date_of_birth | s/-.*$//) * 1
         : $$1::natus          ? ($$1::natus | s/-.*$//) * 1
         : $$1::nata           ? ($$1::nata | s/-.*$//) * 1
         : (($$1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | %(id = ::PassengerId * 1; ::Sex;
     pclass = ::Pclass * 1; survived = ::Survived * 1;
     age = (( ::Age ? ::Age * 1 : 1912 - $.born ));
     age_source = (( ::Age ? "manifest" : $.born ? "records" : "unknown" )))
""")
df = full.df
print(df.shape)
print(df.groupby("age_source")["survived"].mean().round(3))

(891, 6)
age_source
manifest    0.406
records     0.318
unknown     0.100
Name: survived, dtype: float64


From 177 unknown ages to 20 — and the recovered rows survived at
32% against 41% for the rows the manifest filled in: the
missingness was never random, and the median would have buried
that. The vocabulary accretes, the queries stay speakable, and the
table you finally export has earned its shape.